# Claims Analysis

In [1]:
# import mods
import pandas as pd
from zipfile import ZipFile 

In [2]:
# export files from zip
file_names = list()
with ZipFile('data/healthcare-claims-where-is-the-money-going-tvhlq.zip', mode = 'r') as zip:
    zip.extractall('data')
    for file in zip.infolist():
        file_names.append(file.filename)
        print(file.filename, file.compress_size, file.file_size)

members.csv 1079 3232
claims.csv 6568 25743


In [3]:
# import csv files to dataframes
members = pd.read_csv('data/members.csv')
claims = pd.read_csv('data/claims.csv')

### Assess members data

In [4]:
members.head()

,member_id,member_age,member_gender,plan_type,enrollment_start_date,enrollment_end_date
0,1,45,F,PPO,5/14/2021,11/30/2023
1,2,52,M,HMO,3/22/2020,8/15/2022
2,3,38,F,EPO,1/10/2022,4/1/2024
3,4,61,M,POS,9/5/2020,12/1/2022
4,5,29,F,PPO,2/28/2023,NaN


In [5]:
#members.describe()
members.info()

<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 6 columns):
 #   Column                 Non-Null Count  Dtype
---  ------                 --------------  -----
 0   member_id              100 non-null    int64
 1   member_age             100 non-null    int64
 2   member_gender          100 non-null    str  
 3   plan_type              100 non-null    str  
 4   enrollment_start_date  100 non-null    str  
 5   enrollment_end_date    85 non-null     str  
dtypes: int64(2), str(4)
memory usage: 4.8 KB


In [6]:
members.nunique(dropna = False)

member_id                100
member_age                36
member_gender              2
plan_type                  4
enrollment_start_date     83
enrollment_end_date       71
dtype: int64

In [7]:
members[['enrollment_start_date', 'enrollment_end_date']] = members[['enrollment_start_date', 'enrollment_end_date']].apply(pd.to_datetime, format = '%m/%d/%Y')
members.info()
members.head(2)

<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 6 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   member_id              100 non-null    int64         
 1   member_age             100 non-null    int64         
 2   member_gender          100 non-null    str           
 3   plan_type              100 non-null    str           
 4   enrollment_start_date  100 non-null    datetime64[us]
 5   enrollment_end_date    85 non-null     datetime64[us]
dtypes: datetime64[us](2), int64(2), str(2)
memory usage: 4.8 KB


,member_id,member_age,member_gender,plan_type,enrollment_start_date,enrollment_end_date
0,1,45,F,PPO,2021-05-14,2023-11-30
1,2,52,M,HMO,2020-03-22,2022-08-15


##### Notes
* 100 members, most values are not null. 
* There are only 15 currently active members (based on # of records without end dates)

To do list
1. ~~Convert enrollment_start_date and enrollment_end_date to datetime~~

### Assess Claims Data

In [8]:
claims.head()

,claim_id,member_id,provider_id,claim_date,claim_type,cpt_code,icd_code,billed_amount,paid_amount
0,1,1,PRV00001,3/12/2023,Outpatient,12345,A12.3,1800.0,1600.0
1,2,1,PRV00002,7/22/2023,Inpatient,67890,B99.4,32000.0,28000.0
2,3,1,PRV00003,1/15/2024,Lab,23456,C50.1,450.0,400.0
3,4,1,PRV00004,9/5/2023,Pharmacy,34567,D12.7,120.0,120.0
4,5,2,PRV00123,4/12/2023,Outpatient,12345,A12.3,2100.0,1680.0


In [9]:
claims.info()

<class 'pandas.DataFrame'>
RangeIndex: 449 entries, 0 to 448
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   claim_id       449 non-null    int64  
 1   member_id      449 non-null    int64  
 2   provider_id    449 non-null    str    
 3   claim_date     449 non-null    str    
 4   claim_type     449 non-null    str    
 5   cpt_code       449 non-null    str    
 6   icd_code       449 non-null    str    
 7   billed_amount  449 non-null    float64
 8   paid_amount    449 non-null    float64
dtypes: float64(2), int64(2), str(5)
memory usage: 31.7 KB


In [10]:
claims.nunique()

claim_id         449
member_id        100
provider_id      103
claim_date       144
claim_type         5
cpt_code         134
icd_code         117
billed_amount     96
paid_amount      152
dtype: int64

In [11]:
# convert claim date to date time
claims[['claim_date']] = claims[['claim_date']].apply(pd.to_datetime, format = '%m/%d/%Y')
claims.info()

<class 'pandas.DataFrame'>
RangeIndex: 449 entries, 0 to 448
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   claim_id       449 non-null    int64         
 1   member_id      449 non-null    int64         
 2   provider_id    449 non-null    str           
 3   claim_date     449 non-null    datetime64[us]
 4   claim_type     449 non-null    str           
 5   cpt_code       449 non-null    str           
 6   icd_code       449 non-null    str           
 7   billed_amount  449 non-null    float64       
 8   paid_amount    449 non-null    float64       
dtypes: datetime64[us](1), float64(2), int64(2), str(4)
memory usage: 31.7 KB


In [12]:
# check for duplicate entries
claims[claims.duplicated(subset = claims.drop(columns = ['claim_id']), keep = False)]

,claim_id,member_id,provider_id,claim_date,claim_type,cpt_code,icd_code,billed_amount,paid_amount


In [13]:
members.describe()
claims.describe()

,claim_id,member_id,claim_date,billed_amount,paid_amount
count,449.000000,449.000000,449,449.000000,449.000000
mean,236.006682,51.654788,2023-08-22 18:45:42.093541,4592.795212,3453.774855
min,1.000000,1.000000,2022-11-03 00:00:00,60.000000,0.000000
25%,123.000000,27.000000,2023-04-15 00:00:00,260.000000,210.000000
50%,240.000000,52.000000,2023-08-02 00:00:00,1800.000000,1280.000000
75%,352.000000,78.000000,2024-01-08 00:00:00,5400.000000,4200.000000
max,464.000000,100.000000,2024-11-18 00:00:00,42000.000000,35000.000000
std,134.396310,29.337381,NaN,6747.042564,5040.669863


##### Notes
* 449 claims
* most data types look appropriate
* no clear misspellings to clean up

To do list
* ~~Convert claim date to datetime~~
* ~~Check for duplicate claims (same member id, provider id and claim date)~~

# Join data

In [14]:
# combine claims and members data on member id
joinedData = claims.merge(members, on = 'member_id', how = 'inner', suffixes = ('_c', '_m'))
joinedData.head()

,claim_id,member_id,provider_id,claim_date,claim_type,cpt_code,icd_code,billed_amount,paid_amount,member_age,member_gender,plan_type,enrollment_start_date,enrollment_end_date
0,1,1,PRV00001,2023-03-12,Outpatient,12345,A12.3,1800.0,1600.0,45,F,PPO,2021-05-14,2023-11-30
1,2,1,PRV00002,2023-07-22,Inpatient,67890,B99.4,32000.0,28000.0,45,F,PPO,2021-05-14,2023-11-30
2,3,1,PRV00003,2024-01-15,Lab,23456,C50.1,450.0,400.0,45,F,PPO,2021-05-14,2023-11-30
3,4,1,PRV00004,2023-09-05,Pharmacy,34567,D12.7,120.0,120.0,45,F,PPO,2021-05-14,2023-11-30
4,5,2,PRV00123,2023-04-12,Outpatient,12345,A12.3,2100.0,1680.0,52,M,HMO,2020-03-22,2022-08-15


In [15]:
joinedData.to_pickle('postMerge.pkl')
joinedData.to_excel('claimsData.xlsx', index = False)

# Explore Data

### Claim Type Cost Breakdown

In [16]:
# Claim Type Cost Breakdown
claim_summary = joinedData.groupby('claim_type').agg(
                    Total_Billed = ('billed_amount', 'sum'),
                    Total_Paid = ('paid_amount', 'sum'),
                    Number_of_Claims = ('claim_id', 'count'))

In [17]:
# Rank claim types by Total paid and sort by rank
claim_summary['claim_rank'] = claim_summary.Total_Paid.rank(ascending = False)
claim_summary.sort_values('claim_rank', ascending = True)

,Total_Billed,Total_Paid,Number_of_Claims,claim_rank
claim_type,,,,
Inpatient,1478601.25,1092456.00,99,1.0
Emergency,384241.55,294441.36,88,2.0
Outpatient,160717.75,129052.75,105,3.0
Lab,25789.90,23412.35,76,4.0
Pharmacy,12814.60,11382.45,81,5.0


##### Notes
* Inpatient is nearly 5 times the amount billed and about 3 times the amount paid than Emergency claim types

### CPT & ICD Cost Drivers

In [18]:
# Top 10 most costly CPT codes 
joinedData.groupby('cpt_code').agg(
    Total_Paid = ('paid_amount', 'sum'),
    Claim_Count = ('claim_id', 'count')).sort_values('Total_Paid', ascending = False).head(10)

,Total_Paid,Claim_Count
cpt_code,,
67890,242735.00,25
23456,203790.75,29
00123,122010.00,12
12345,115236.90,47
99223,57350.00,6
34567,54445.00,23
45678,52054.20,21
00567,38969.00,9
54321,36600.00,11


In [19]:
# which procedures are the most costly?
cpt_summary = joinedData.groupby('cpt_code').agg(
                                            Total_Paid = ('paid_amount', 'sum'),
                                            Claim_count = ('claim_id', 'count'))

cpt_summary['average_paid_per_claim'] = round(cpt_summary.Total_Paid / cpt_summary.Claim_count, 2)
cpt_summary.sort_values('average_paid_per_claim', ascending = False).head(10)
# Note -- the tenth row is the 3rd highest total paid cpt code. It's a higher cost procedure and many (12) have been performed. Most of the other procedures in this list are high cost, but have only been performed once.

,Total_Paid,Claim_count,average_paid_per_claim
cpt_code,,,
36512,25600.0,1,25600.0
10101,17250.6,1,17250.6
89012,15250.0,1,15250.0
00512,14000.0,1,14000.0
50255,13000.0,1,13000.0
20610,12000.0,1,12000.0
61234,10500.0,1,10500.0
43752,10200.0,1,10200.0
99217,10200.0,1,10200.0


In [20]:
# which codes have the largest # of claims?
joinedData.groupby('cpt_code').agg(Claim_Count = ('claim_id', 'count')).sort_values('Claim_Count', ascending = False).head()
# Note -- the 5 highest claim counts are found within the 7 highest total paid procedures

,Claim_Count
cpt_code,
12345,47
23456,29
67890,25
34567,23
45678,21


In [21]:
# Top 10 most costly ICD codes 
joinedData.groupby('icd_code').agg(
    Total_Paid = ('paid_amount', 'sum'),
    Claim_count = ('claim_id', 'count')).sort_values('Total_Paid', ascending = False).head(10)
# Note -- the 5 codes with the highest # of claims are in the 10 highest cost diagnoses 

,Total_Paid,Claim_count
icd_code,,
I10,259566.00,34
A12.3,152147.00,42
B20,140990.00,17
B20.1,105210.00,13
C34.91,62905.00,18
B99.4,51000.00,4
E11.65,50512.00,11
J45.909,44465.56,17
E11.9,34766.00,51


In [22]:
# which codes have the largest # of claims?
joinedData.groupby('icd_code').agg(Claim_Count = ('claim_id', 'count')).sort_values('Claim_Count', ascending = False).head()

,Claim_Count
icd_code,
E11.9,51
A12.3,42
I10,34
C34.91,18
B20,17


### Member-Level analysis

In [23]:
# total paid per member
joinedData.groupby('member_id').agg(
    Total_Paid = ('paid_amount', 'sum')).sort_values('Total_Paid', ascending = False)

,Total_Paid
member_id,
6,43300.00
32,40080.00
58,35920.00
82,30690.00
28,30560.00
...,...
76,5080.00
51,4290.00
34,2540.00


In [24]:
# top 5-10 highest cost members
memberCost = joinedData.groupby('member_id').agg(
    Total_Paid = ('paid_amount', 'sum')).sort_values('Total_Paid', ascending = False).head(10)
memberCost.reset_index()

,member_id,Total_Paid
0,6,43300.00
1,32,40080.00
2,58,35920.00
3,82,30690.00
4,28,30560.00
5,20,30375.00
6,71,30230.00
7,1,30120.00
8,36,25530.75
9,8,23046.80


### OTHER

In [26]:
joinedData.describe()

,claim_id,member_id,claim_date,billed_amount,paid_amount,member_age,enrollment_start_date,enrollment_end_date
count,449.000000,449.000000,449,449.000000,449.000000,449.000000,449,381
mean,236.006682,51.654788,2023-08-22 18:45:42.093541,4592.795212,3453.774855,48.064588,2021-06-24 01:04:08.552338,2023-02-09 10:53:51.496063
min,1.000000,1.000000,2022-11-03 00:00:00,60.000000,0.000000,21.000000,2018-03-12 00:00:00,2020-08-19 00:00:00
25%,123.000000,27.000000,2023-04-15 00:00:00,260.000000,210.000000,37.000000,2020-09-01 00:00:00,2022-06-30 00:00:00
50%,240.000000,52.000000,2023-08-02 00:00:00,1800.000000,1280.000000,45.000000,2021-03-15 00:00:00,2023-03-14 00:00:00
75%,352.000000,78.000000,2024-01-08 00:00:00,5400.000000,4200.000000,58.000000,2022-05-10 00:00:00,2023-12-31 00:00:00
max,464.000000,100.000000,2024-11-18 00:00:00,42000.000000,35000.000000,94.000000,2023-08-09 00:00:00,2025-03-15 00:00:00
std,134.396310,29.337381,NaN,6747.042564,5040.669863,14.825859,NaN,NaN
